# FreshGuard — Notebook 1 / 5: Dataset Audit, Dedup, and Split

**Purpose.** Take the Kaggle `ulnnproject/food-freshness-dataset` (a merge of three source datasets) and turn it into a single frozen `manifest.parquet` with cluster‑disjoint train/val/test splits.

**Why this matters.** The previous project iteration reported 0.97 classifier accuracy but explicitly skipped perceptual‑hashing dedup (`PRD.md:76`). The dataset is a 3‑source merge plus an internal Bellpepper/Capciscum duplication, so the same image very likely appears in multiple folders. Without dedup, those duplicates leak across train/val/test and inflate the metrics. This notebook fixes that.

**Outputs (publish as a Kaggle Dataset called `freshguard-manifest`):**
- `manifest.parquet` — one row per usable image: `path, produce_type, freshness, combined_label, cluster_id, split, width, height, sample_weight, phash`
- `class_weights.json` — inverse‑frequency weights for the 24 fine‑grained classes
- `audit_report.md` — per‑class counts, dedup stats, decision log

**Expected runtime.** ≈20–40 minutes on a Kaggle CPU instance for the full 70k+ images (image decode + phash dominate).

---

**Class contract.** 12 produce types × {fresh, rotten} = **24 classes**. The Kaggle dataset's `Bellpepper` and `Capciscum` folders both map to the canonical `bellpepper` class (user‑verified the same vegetable). Folder typos `bittergroud` and `okara` are normalized to `bitter_gourd` and `okra`.

In [ ]:
!pip install --quiet imagehash pybktree pyarrow

In [ ]:
from __future__ import annotations

import json
import re
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from pathlib import Path

import imagehash
import numpy as np
import pandas as pd
import pybktree
from PIL import Image, ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = False  # we want corrupt files to fail loudly

# ---------------------------------------------------------------
# Configuration — edit these if your Kaggle dataset path differs.
# ---------------------------------------------------------------
DATASET_ROOT = Path("/kaggle/input/datasets/ulnnproject/food-freshness-dataset/Dataset")
OUTPUT_DIR = Path("/kaggle/working")
PHASH_HAMMING_THRESHOLD = 5  # near-duplicate threshold (out of 64 bits)
TRAIN_FRAC, VAL_FRAC, TEST_FRAC = 0.70, 0.15, 0.15
RNG_SEED = 0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not DATASET_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATASET_ROOT}. "
        "Attach `ulnnproject/food-freshness-dataset` as an input and update DATASET_ROOT if the path differs."
    )

print(f"Dataset root: {DATASET_ROOT}")
print(f"Output dir:   {OUTPUT_DIR}")

## Inline label normalization

Mirrors `src/freshness/utils/labels.py` so this notebook runs without needing to `pip install` the project from GitHub. If you change the canonical class set in the package, update this cell to match.

**Canonical 12 produce types:** apple, banana, **bellpepper** (= bellpepper + capciscum + capsicum), bitter_gourd (= bittergroud), carrot, cucumber, mango, **okra** (= okara), orange, potato, strawberry, tomato.

In [ ]:
PRODUCE_TYPES = (
    "apple", "banana", "bellpepper", "bitter_gourd", "carrot", "cucumber",
    "mango", "okra", "orange", "potato", "strawberry", "tomato",
)
FRESHNESS_LEVELS = ("fresh", "rotten")
COMBINED_CLASSES = tuple(f"{t}_{f}" for t in PRODUCE_TYPES for f in FRESHNESS_LEVELS)
assert len(COMBINED_CLASSES) == 24

TYPE_ALIASES = {
    "apple": "apple", "apples": "apple",
    "banana": "banana", "bananas": "banana",
    # Bellpepper + Capciscum/Capsicum collapse to one canonical class.
    "bellpepper": "bellpepper", "bellpeppers": "bellpepper",
    "bell_pepper": "bellpepper", "bell_peppers": "bellpepper",
    "pepper": "bellpepper", "peppers": "bellpepper",
    "capsicum": "bellpepper", "capsicums": "bellpepper",
    "capciscum": "bellpepper", "capciscums": "bellpepper",
    "bittergourd": "bitter_gourd", "bittergourds": "bitter_gourd",
    "bitter_gourd": "bitter_gourd", "bitter_gourds": "bitter_gourd",
    "bittergroud": "bitter_gourd", "bittergrounds": "bitter_gourd",
    "carrot": "carrot", "carrots": "carrot",
    "cucumber": "cucumber", "cucumbers": "cucumber",
    "mango": "mango", "mangos": "mango", "mangoes": "mango",
    "okra": "okra", "okras": "okra", "okara": "okra", "okaras": "okra",
    "orange": "orange", "oranges": "orange",
    "potato": "potato", "potatoes": "potato",
    "strawberry": "strawberry", "strawberries": "strawberry",
    "tomato": "tomato", "tomatoes": "tomato",
}

FRESHNESS_ALIASES = {
    "fresh": "fresh", "purefresh": "fresh", "good": "fresh",
    "healthy": "fresh", "ripe": "fresh", "normal": "fresh", "unripe": "fresh",
    "rotten": "rotten", "rotton": "rotten", "rot": "rotten",
    "stale": "rotten", "spoiled": "rotten", "spoilt": "rotten",
    "bad": "rotten", "moldy": "rotten", "mouldy": "rotten",
    "decayed": "rotten", "damaged": "rotten", "defective": "rotten",
    "diseased": "rotten", "unhealthy": "rotten", "overripe": "rotten",
}

_TOKEN_RE = re.compile(r"[^a-z0-9]+")

def _collapse(value: str) -> str:
    return "".join(t for t in _TOKEN_RE.split(value.lower()) if t)

def normalize_type(value: str) -> str | None:
    c = _collapse(value)
    if c in TYPE_ALIASES:
        return TYPE_ALIASES[c]
    for alias, canon in sorted(TYPE_ALIASES.items(), key=lambda kv: -len(kv[0])):
        if alias in c:
            return canon
    return None

def normalize_freshness(value: str) -> str | None:
    c = _collapse(value)
    if c in FRESHNESS_ALIASES:
        return FRESHNESS_ALIASES[c]
    for alias, canon in sorted(FRESHNESS_ALIASES.items(), key=lambda kv: -len(kv[0])):
        if c.startswith(alias):
            return canon
    return None

def infer_labels_from_path(path: Path) -> tuple[str, str] | None:
    """Return (produce_type, freshness) inferred from any folder/stem in the path, else None."""
    candidates = [path.stem, *reversed(path.parts[:-1])]
    produce_type = None
    freshness = None
    for candidate in candidates:
        if produce_type is None:
            produce_type = normalize_type(candidate)
        if freshness is None:
            freshness = normalize_freshness(candidate)
        if produce_type and freshness:
            return produce_type, freshness
    return None

# Smoke test
assert infer_labels_from_path(Path("Dataset/FreshCapciscum/img.jpg")) == ("bellpepper", "fresh")
assert infer_labels_from_path(Path("Dataset/RottenBittergroud/img.jpg")) == ("bitter_gourd", "rotten")
assert infer_labels_from_path(Path("Dataset/FreshOkara/img.jpg")) == ("okra", "fresh")
print("Label normalization smoke test passed.")

## Step 1 — Walk dataset and decode each image

Recursively glob the dataset, infer `(produce_type, freshness)` from each path, and verify each image actually decodes. Corrupt or unreadable files are dropped (and counted).

In [ ]:
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def _decode_record(path_str: str) -> dict:
    path = Path(path_str)
    try:
        with Image.open(path) as img:
            img.load()  # force decode
            width, height = img.size
        return {"path": path_str, "width": width, "height": height, "ok": True, "error": ""}
    except Exception as exc:
        return {"path": path_str, "width": 0, "height": 0, "ok": False, "error": str(exc)[:120]}

all_paths = [
    p for p in DATASET_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES
]
print(f"Found {len(all_paths):,} candidate image files.")

with ProcessPoolExecutor() as pool:
    decoded = list(pool.map(_decode_record, [str(p) for p in all_paths], chunksize=64))

decode_df = pd.DataFrame(decoded)
corrupt_df = decode_df[~decode_df.ok].copy()
ok_df = decode_df[decode_df.ok].copy()
print(f"Decoded OK: {len(ok_df):,}")
print(f"Corrupt:    {len(corrupt_df):,}")
if not corrupt_df.empty:
    display(corrupt_df.head(10))

In [ ]:
# Attach labels using the inline normalizer.
rows = []
missing_type = []
for _, row in ok_df.iterrows():
    labels = infer_labels_from_path(Path(row.path))
    if labels is None:
        missing_type.append(row.path)
        continue
    produce_type, freshness = labels
    rows.append({
        "path": row.path,
        "produce_type": produce_type,
        "freshness": freshness,
        "combined_label": f"{produce_type}_{freshness}",
        "width": row.width,
        "height": row.height,
    })

manifest = pd.DataFrame(rows)
print(f"Labelled rows:        {len(manifest):,}")
print(f"Could not infer type: {len(missing_type):,}")
print("Per-class counts (sanity check vs Kaggle dataset card):")
display(
    manifest.groupby(["produce_type", "freshness"]).size().unstack(fill_value=0)
)

## Step 2 — Compute perceptual hash for every image

We use 8×8 DCT perceptual hashing (`imagehash.phash`) which yields a 64‑bit fingerprint per image. Hamming distance ≤ 5 between two phashes (out of 64 bits) is a robust near‑duplicate threshold across the literature.

In [ ]:
def _phash_one(path_str: str) -> tuple[str, int]:
    with Image.open(path_str) as img:
        h = imagehash.phash(img.convert("RGB"))
    # Convert to a uint64 integer for fast XOR-based Hamming distance later.
    bits = "".join("1" if b else "0" for row in h.hash for b in row)
    return path_str, int(bits, 2)

with ProcessPoolExecutor() as pool:
    phash_results = dict(pool.map(_phash_one, manifest.path.tolist(), chunksize=64))

manifest["phash"] = manifest.path.map(phash_results)
manifest["phash_hex"] = manifest.phash.map(lambda x: f"{x:016x}")
print(f"Computed {manifest.phash.notna().sum():,} phashes.")

## Step 3 — Cluster near‑duplicates with a BK‑tree

BK‑trees give us efficient `O(log n)` average‑case Hamming distance queries. We build the tree from all phashes, query each phash for neighbors within Hamming ≤ `PHASH_HAMMING_THRESHOLD`, then run union‑find on the resulting edges to assign every image a `cluster_id`. All near‑duplicates of one image end up in the same cluster.

Given the dataset is a 3‑source merge plus the Bellpepper/Capciscum duplication, we expect a non‑trivial number of multi‑image clusters.

In [ ]:
def hamming(a: int, b: int) -> int:
    return (a ^ b).bit_count()

phashes = manifest.phash.tolist()
tree = pybktree.BKTree(hamming, phashes)

# Build edges: (i, j) where i < j and hamming(phash_i, phash_j) <= threshold.
phash_to_indices: dict[int, list[int]] = defaultdict(list)
for i, h in enumerate(phashes):
    phash_to_indices[h].append(i)

edges: list[tuple[int, int]] = []
seen_query = set()
for i, h in enumerate(phashes):
    if h in seen_query:
        continue
    seen_query.add(h)
    neighbors = tree.find(h, PHASH_HAMMING_THRESHOLD)
    src_indices = phash_to_indices[h]
    for _dist, neighbor_h in neighbors:
        if neighbor_h == h:
            continue
        for a in src_indices:
            for b in phash_to_indices[neighbor_h]:
                if a < b:
                    edges.append((a, b))
    # Also link images that share an exact phash.
    if len(src_indices) > 1:
        first = src_indices[0]
        for other in src_indices[1:]:
            edges.append((first, other))

print(f"Edges (near-duplicate pairs): {len(edges):,}")

# Union-find
parent = list(range(len(manifest)))
def find(x: int) -> int:
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a: int, b: int) -> None:
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
for a, b in edges:
    union(a, b)

manifest["cluster_id"] = [find(i) for i in range(len(manifest))]
n_clusters = manifest.cluster_id.nunique()
print(f"Clusters: {n_clusters:,}")
print(f"Dedup ratio (clusters / images): {n_clusters / len(manifest):.4f}")

cluster_size_hist = manifest.groupby("cluster_id").size().value_counts().sort_index()
print("Cluster size histogram (size → count of clusters):")
display(cluster_size_hist.head(15))

## Step 4 — Stratified, cluster‑disjoint train/val/test split

We split at the **cluster level**, not the image level, so that no image and its near‑duplicate ever land in different splits. Within each `(produce_type, freshness)` stratum we shuffle the clusters and assign them greedily to splits to hit the target ratios as closely as possible, accounting for cluster sizes.

In [ ]:
rng = np.random.default_rng(RNG_SEED)

# Decide a single (produce_type, freshness) for each cluster: the modal label
# of its members. Multi-label clusters (rare — only when phash collides on
# different produce types) get assigned by majority vote.
cluster_label = (
    manifest.groupby("cluster_id")["combined_label"]
    .agg(lambda s: s.mode().iat[0])
    .to_dict()
)
cluster_size = manifest.groupby("cluster_id").size().to_dict()

split_assignment: dict[int, str] = {}
split_targets = {"train": TRAIN_FRAC, "val": VAL_FRAC, "test": TEST_FRAC}

for label, group in manifest.groupby("combined_label"):
    cluster_ids = sorted(set(group.cluster_id))
    rng.shuffle(cluster_ids)
    total_images = sum(cluster_size[c] for c in cluster_ids)
    targets = {s: int(round(total_images * frac)) for s, frac in split_targets.items()}
    # Force at least 1 image per split for minorities (Bittergroud has only ~684 total).
    for s in ("val", "test"):
        if targets[s] == 0 and total_images >= 3:
            targets[s] = 1
            targets["train"] = max(0, targets["train"] - 1)
    placed = {s: 0 for s in split_targets}
    for cid in cluster_ids:
        # Place this cluster in the split with the largest remaining deficit.
        deficits = {s: targets[s] - placed[s] for s in split_targets}
        best = max(deficits, key=deficits.get)
        split_assignment[cid] = best
        placed[best] += cluster_size[cid]

manifest["split"] = manifest.cluster_id.map(split_assignment)

# Sanity checks
split_overlap = (
    manifest.groupby("cluster_id")["split"].nunique().eq(1).all()
)
print(f"Cluster-disjoint splits: {split_overlap}")
assert split_overlap, "BUG: some cluster spans multiple splits"

split_summary = (
    manifest.groupby(["combined_label", "split"]).size().unstack(fill_value=0)
)
split_summary["total"] = split_summary.sum(axis=1)
for s in ("train", "val", "test"):
    if s in split_summary.columns:
        split_summary[f"{s}_pct"] = (split_summary[s] / split_summary.total * 100).round(1)
display(split_summary)

print("Overall split sizes:")
print(manifest.split.value_counts())

## Step 5 — Per‑class sample weights

Inverse‑frequency weights, normalized so the mean weight is 1. The classifier training notebook (notebook 4) uses `sample_weight` for `WeightedRandomSampler` and the per‑class array for `CrossEntropyLoss(weight=...)`.

In [ ]:
train_counts = (
    manifest[manifest.split == "train"]
    .combined_label.value_counts()
    .reindex(COMBINED_CLASSES, fill_value=0)
)
raw_weights = 1.0 / train_counts.replace(0, np.nan)
raw_weights = raw_weights.fillna(raw_weights.max())  # any zero-count class gets max weight
class_weights = (raw_weights / raw_weights.mean()).round(4)
weight_dict = class_weights.to_dict()

manifest["sample_weight"] = manifest.combined_label.map(weight_dict).astype(float)

print("Class weights (sorted by weight, descending — minority classes first):")
display(class_weights.sort_values(ascending=False).to_frame("weight"))

max_min_ratio = class_weights.max() / class_weights[class_weights > 0].min()
print(f"Largest:smallest weight ratio: {max_min_ratio:.1f} (matches dataset imbalance)")

## Step 6 — Save outputs

Three files land in `/kaggle/working/`. After the run, click **Save Version** → **Save & Run All**, then publish the working directory as a Kaggle Dataset (e.g. `your-handle/freshguard-manifest`). The next four notebooks add this dataset as an input.

In [ ]:
manifest_out = manifest[[
    "path", "produce_type", "freshness", "combined_label",
    "cluster_id", "split", "width", "height", "sample_weight", "phash_hex",
]].copy()
manifest_out.to_parquet(OUTPUT_DIR / "manifest.parquet", index=False)

with (OUTPUT_DIR / "class_weights.json").open("w") as fh:
    json.dump(
        {
            "classes": list(COMBINED_CLASSES),
            "weights": [float(weight_dict[c]) for c in COMBINED_CLASSES],
            "train_counts": {c: int(train_counts[c]) for c in COMBINED_CLASSES},
        },
        fh,
        indent=2,
    )

report_lines = [
    "# FreshGuard Dataset Audit Report",
    "",
    f"- Source dataset root: `{DATASET_ROOT}`",
    f"- Candidate files: **{len(all_paths):,}**",
    f"- Decoded OK: **{len(ok_df):,}**",
    f"- Corrupt: **{len(corrupt_df):,}**",
    f"- Could not infer label: **{len(missing_type):,}**",
    f"- Final usable rows: **{len(manifest):,}**",
    f"- Distinct combined labels: **{manifest.combined_label.nunique()}** (expected 24)",
    "",
    "## Dedup",
    "",
    f"- Phash bits: 64 (8×8 DCT)",
    f"- Hamming threshold: ≤ {PHASH_HAMMING_THRESHOLD}",
    f"- Near-duplicate edges: **{len(edges):,}**",
    f"- Clusters: **{n_clusters:,}** (dedup ratio {n_clusters / len(manifest):.4f})",
    "",
    "## Bellpepper / Capciscum merge",
    "",
    "User-verified that `Bellpepper` and `Capciscum` source folders contain the same vegetable.",
    "Both folders are normalized into the canonical `bellpepper` class. Final class count: 24.",
    "",
    "## Folder typo normalizations",
    "",
    "- `Bittergroud` → `bitter_gourd`",
    "- `Okara` → `okra` (the dataset folder name is a typo — okara is fermented soybean pulp, a different food)",
    "",
    "## Per-class split counts",
    "",
    split_summary.to_markdown(),
    "",
    "## Class weights (largest = minority class)",
    "",
    class_weights.sort_values(ascending=False).to_frame("weight").to_markdown(),
]
(OUTPUT_DIR / "audit_report.md").write_text("\n".join(report_lines))

print("Wrote:")
for name in ("manifest.parquet", "class_weights.json", "audit_report.md"):
    p = OUTPUT_DIR / name
    print(f"  {p} ({p.stat().st_size / 1024:.1f} KiB)")

## Publish as a Kaggle Dataset

1. Click **Save Version** → **Save & Run All (Commit)** in the top right.
2. After the commit succeeds, open this notebook's **Output** tab.
3. Click **New Dataset**, name it `freshguard-manifest`, set visibility to Private (or Public if you want others to reproduce).
4. The next four notebooks attach this dataset as an input via **Add Input** → search `freshguard-manifest`.

**Verification before publishing:**
- `manifest.parquet` row count is in the expected ballpark (≈70k after dedup).
- The `Cluster-disjoint splits: True` assertion passed.
- The per‑class split table shows every minority class (Bittergroud, Okara, Mango, Strawberry) present in val and test.